In [ ]:
import pandas as pd
from sklearn.preprocessing import OrdinalEncoder, StandardScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.model_selection import train_test_split
from sklearn import model_selection
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

In [ ]:
tree_data = pd.read_csv(r'..\Data\new_york_tree_census_2015.csv')

In [ ]:
tree_data_small = tree_data[['tree_dbh', 'curb_loc', 'health', 'spc_common', 'sidewalk', 'zipcode']].copy()
tree_data_small

In [ ]:
# initial clean of the data - drop na
tree_data_small.dropna(inplace=True)

tree_data_small.info()

In [ ]:
def get_ratio(series):
       val_counts = series.value_counts()
       if len(val_counts) > 1:
              return val_counts.iloc[0]/ val_counts.iloc[1]
       # if there is only one value then the ratio is 1
       return 1
           
def get_first(series):
       val_counts = series.value_counts()
       return val_counts.index[0]

def get_second(series):
       val_counts = series.value_counts()
       if len(val_counts) > 1:
              return val_counts.index[1]
       else:
              return val_counts.index[-1]

def get_third(series):
       val_counts = series.value_counts()
       if len(val_counts) > 2:
              return val_counts.index[2]
       else:
              return val_counts.index[-1]
       
def count_unique_vals(series):
       return series.value_counts().to_dict()
       

In [ ]:
# because we are going to join on zip code, we need to make meaningful aggregates for each column per zip code
tree_data_agg = tree_data_small.groupby('zipcode').agg(
       dbh_mean=('tree_dbh', 'mean'),
       curb_ratio=('curb_loc', get_ratio),
       sidewalk_ratio=('sidewalk', get_ratio),
       health_counts=('health', count_unique_vals),
       species_1=('spc_common', get_first),
       species_2=('spc_common', get_second),
       species_3=('spc_common', get_third),
       tree_count=('tree_dbh', 'size')
    
).reset_index()

health_counts_df = pd.DataFrame(tree_data_agg['health_counts'].tolist()).fillna(0).astype(int)
tree_data_agg = pd.concat([tree_data_agg, health_counts_df], axis=1).drop(columns=['health_counts'])

tree_data_agg


In [ ]:
# need to encode categorical data
encoder = OrdinalEncoder()

# encode the three species columns
tree_encoded_df = tree_data_agg
tree_encoded_df[['species_1', 'species_2', 'species_3']] = encoder.fit_transform(tree_data_agg[['species_1', 'species_2', 'species_3']])

tree_encoded_df

In [ ]:
crime_data = pd.read_csv(r'..\Data\2015_Crime.csv', low_memory=False)

In [ ]:
crime_data.head()
# Keep only the specified columns
crime_data = crime_data[['Violation Date', 'Violation Time', 'Issuing Agency',
                         'Violation Location (Zip Code)', 
                         'Penalty Imposed', 'Charge #1: Code', 
                         'Charge #2: Code', 'Charge #3: Code', 'Charge #4: Code', 
                         'Charge #5: Code', 'Charge #6: Code', 'Charge #7: Code', 
                         'Charge #8: Code', 'Charge #9: Code', 'Charge #10: Code']]

# Remove rows where the 'Violation Location (Zip Code)' column is NaN
crime_data = crime_data.dropna(subset=['Violation Location (Zip Code)'])

crime_data = crime_data.dropna(subset=['Issuing Agency'])


In [ ]:
# Extract columns containing charges
charge_columns = [col for col in crime_data.columns if "Charge" in col]

crime_data = crime_data.melt(
    id_vars=[col for col in crime_data.columns if col not in charge_columns],  # Keep these columns unchanged  # Columns to unpivot
    var_name="Original Charge Column",  # New column for original charge column names
    value_name="Charge: Code",  # New column for charge values
)

# Drop rows where Charge: Code is NaN
crime_data = crime_data.dropna(subset=["Charge: Code"]).drop(columns=["Original Charge Column"])

# Reset index for a clean DataFrame
crime_data = crime_data.reset_index(drop=True)

# because we are going to join on zip code, we need to make meaningful aggregates for each column per zip code
crime_data_final = crime_data.groupby('Violation Location (Zip Code)').agg(
    issuing_agency_1=('Issuing Agency', get_first),
    issuing_agency_2=('Issuing Agency', get_second),
    issuing_agency_3=('Issuing Agency', get_third),
    penalty_imposed=('Penalty Imposed', 'mean'),
    charge_1 = ('Charge: Code', get_first),
    charge_2 = ('Charge: Code', get_second),
    charge_3 = ('Charge: Code', get_third),
    crime_count = ('Issuing Agency', 'size')
).reset_index()

# calculating if the crime count is above the mean
crime_data_final['crime_above_avg'] = (crime_data_final['crime_count'] > crime_data_final['crime_count'].mean()).astype(int)

crime_data_final.rename(columns={'Violation Location (Zip Code)': 'zipcode'}, inplace=True)

crime_data_final.drop(crime_data_final[~crime_data_final['zipcode'].astype(str).apply(lambda x: x.isdigit())].index, inplace=True)

print(crime_data_final['zipcode'].unique())

crime_data_final['zipcode'] = crime_data_final['zipcode'].astype(int)

crime_data_final.head()
print(len(crime_data_final['crime_above_avg'] == 0))
print(len(crime_data_final['crime_above_avg'] ==1))

In [ ]:
encoder = OrdinalEncoder()

# encode the three species columns
crime_encoded_df = crime_data_final
crime_encoded_df[['issuing_agency_1', 'issuing_agency_2', 'issuing_agency_3']] = encoder.fit_transform(
    crime_data_final[['issuing_agency_1', 'issuing_agency_2', 'issuing_agency_3']])

crime_encoded_df[['charge_1', 'charge_2', 'charge_3']] = encoder.fit_transform(
    crime_data_final[['charge_1', 'charge_2', 'charge_3']])


In [ ]:
crime_encoded_df

In [ ]:
# combining datasets for ML algs
tree_final = tree_encoded_df
crime_final = crime_encoded_df

merged_df = pd.merge(tree_final, crime_final, on='zipcode', how='inner')

# scaled data
non_scaling_cols = ['zipcode', 'crime_above_avg']
scaling_cols = [col for col in merged_df.columns if col not in non_scaling_cols]
scaler = StandardScaler()
merged_df[scaling_cols] = pd.DataFrame(scaler.fit_transform(merged_df[scaling_cols]))
final_df = merged_df[scaling_cols + non_scaling_cols]

final_df

In [ ]:
# visualize data


sns.scatterplot(x='tree_count', y='penalty_imposed', hue='crime_above_avg', data=final_df)
sns.regplot(x='tree_count', y='penalty_imposed', scatter=False, color='red', data=final_df)
plt.ylabel('Crime Penalty Imposed')
plt.xlabel('Tree Count')
plt.title('Crime penalty imposed vs tree count')
plt.show()

sns.scatterplot(x='tree_count', y='crime_count', hue='crime_above_avg', data=final_df)
sns.regplot(x='tree_count', y='crime_count', scatter=False, color='red', data=final_df)
plt.ylabel('Crime Count')
plt.xlabel('Tree Count')
plt.title('Tree Count vs crime count')
plt.show()

In [ ]:
# ML algs - predicting crime count
from sklearn.metrics import make_scorer, precision_score

#train test split
X = final_df.drop('crime_above_avg', axis=1)
y = final_df['crime_above_avg']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=1/3.)

knn = KNeighborsClassifier(n_neighbors=3)
knn.fit(X_train, y_train)

y_pred = knn.predict(X_test)

print("- 5-Fold Cross Validation -")
precision_scorer = make_scorer(precision_score, average='weighted', zero_division=0)

scores = model_selection.cross_validate(knn, X_test, y_test, cv=5,
                                        scoring={'precision_weighted': precision_scorer, 
                                        'f1_weighted': 'f1_weighted', 
                                        'recall_weighted': 'recall_weighted'})
print("F1: ", f"{np.mean(scores['test_f1_weighted']):.4f}")
print("Precision: ", f"{np.mean(scores['test_precision_weighted']):.4f}")
print("Recall: ", f"{np.mean(scores['test_recall_weighted']):.4f}")

from sklearn.metrics import ConfusionMatrixDisplay, confusion_matrix

# Compute confusion matrix
cm = confusion_matrix(y_test, y_pred, labels=knn.classes_)

# Display confusion matrix
disp = ConfusionMatrixDisplay(confusion_matrix=cm)
disp.plot(cmap='Blues')
plt.title("Confusion Matrix")
plt.show()